# Загрузка данных и обучение модели

In [1]:
import os
from pathlib import Path

clearml_config = Path("../.clearml.conf").resolve()
if clearml_config.is_file():
    os.environ.setdefault("CLEARML_CONFIG_FILE", str(clearml_config))

import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from clearml import Task
from plotly.subplots import make_subplots
from scipy.stats import bootstrap
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve, classification_report,
                             confusion_matrix, f1_score, average_precision_score, log_loss)
from sklearn.model_selection import (StratifiedKFold, cross_val_score, cross_val_predict,
                                     train_test_split)
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests

optuna.logging.set_verbosity(optuna.logging.WARNING)

task = Task.init(
    project_name="P&G AI Citation",
    task_name="EDA and model comparison",
    reuse_last_task_id=False,
)
logger = task.get_logger()

task.connect({
    "data_file": "PG_GEO_Dataset.xlsx",
    "test_size": 0.2,
    "random_state": 42,
    "cv_folds": 3,
    "optuna_trials": 50,
})


ClearML Task: created new task id=d899d5fe526e40f38413920d79a88fa4


2026-11-02 17:52:45,342 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []


======> WARNING! Git diff too large to store (940kb), skipping uncommitted changes <======


ClearML results page: https://app.clear.ml/projects/aaec52044be44d909c87e426dcc33b10/tasks/d899d5fe526e40f38413920d79a88fa4/output/log
2026-11-02 17:52:45,744 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found


{'data_file': 'PG_GEO_Dataset.xlsx',
 'test_size': 0.2,
 'random_state': 42,
 'cv_folds': 3,
 'optuna_trials': 50}

In [2]:
df = pd.read_excel("../data/PG_GEO_Dataset.xlsx")

assert (
    df["is_serp_link"].astype(bool)
    == df["source_type_serp_internal"].eq(1)
).all()

df = df.loc[~df["is_serp_link"].astype(bool)].copy()

print(f'''Размер датасета: {df.shape}
Баланс классов:
{df['cited_in_generative_answer'].value_counts()}
Доля положительных: {df['cited_in_generative_answer'].mean():.2%}''')

logger.report_single_value(name="dataset_rows", value=len(df))
logger.report_single_value(name="dataset_positive_share", value=float(df['cited_in_generative_answer'].mean()))

Размер датасета: (195, 62)
Баланс классов:
cited_in_generative_answer
1    110
0     85
Name: count, dtype: int64
Доля положительных: 56.41%


# Исследовательский анализ данных (EDA)

После исключения внутренних ссылок поисковой выдачи EDA использует 195 страниц и исходные признаки до отбора и разделения на train/test. PCA здесь служит только для визуализации структуры данных.

In [3]:
eda_target = "cited_in_generative_answer"
eda_meta = ["query", "query_type", "product_category", "source_url", "domain", "source_type"]
eda_numeric = df.drop(columns=eda_meta + [eda_target]).select_dtypes(include=["number", "bool"])
eda_unique = eda_numeric.nunique(dropna=True)
eda_multi = eda_unique.index[eda_unique > 2].tolist()
eda_binary = eda_unique.index[eda_unique == 2].tolist()

eda_missing = eda_numeric.isna().sum().loc[lambda s: s > 0].sort_values(ascending=False)

print(f'''Объектов: {len(df)}, признаков: {eda_numeric.shape[1]}
Бинарных: {len(eda_binary)}, с более чем двумя значениями: {len(eda_multi)}
Пропуски в признаках:
{eda_missing.to_string()}
Повторы query + source_url: {df.duplicated(['query', 'source_url']).sum()}''')

logger.report_table(title="EDA", series="missing_values",
                    table_plot=eda_missing.rename_axis("feature").reset_index(name="missing"))


Объектов: 195, признаков: 55
Бинарных: 42, с более чем двумя значениями: 10
Пропуски в признаках:
user_rating     21
review_count    21
Повторы query + source_url: 1


## Неразличимые страницы и противоречивые метки

In [4]:
pattern_counts = (
    df.groupby(eda_numeric.columns.tolist(), dropna=False)[eda_target]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=[0, 1], fill_value=0)
)

conflicting_patterns = pattern_counts.loc[
    (pattern_counts[0] > 0) & (pattern_counts[1] > 0)
]
conflicting_rows = int(conflicting_patterns.sum(axis=1).sum())
minimum_errors = int(conflicting_patterns.min(axis=1).sum())

print(f'''Одинаковые признаки при разных метках:
Наборов признаков: {len(conflicting_patterns)}
Строк в этих наборах: {conflicting_rows} из {len(df)}
Минимум ошибок любой детерминированной модели на этих строках: {minimum_errors}
Максимальная точность на текущих данных при запоминании признаков: {1 - minimum_errors / len(df):.1%}''')

logger.report_single_value(
    name="conflicting_feature_patterns",
    value=len(conflicting_patterns),
)
logger.report_single_value(
    name="rows_with_conflicting_labels",
    value=conflicting_rows,
)
logger.report_single_value(
    name="minimum_errors_from_identical_features",
    value=minimum_errors,
)


Одинаковые признаки при разных метках:
Наборов признаков: 16
Строк в этих наборах: 66 из 195
Минимум ошибок любой детерминированной модели на этих строках: 27
Максимальная точность на текущих данных при запоминании признаков: 86.2%


## Корреляции признаков

In [5]:
eda_corr = eda_numeric.corr()
fig = go.Figure(go.Heatmap(
    z=eda_corr.to_numpy(), x=eda_corr.columns, y=eda_corr.index,
    zmin=-1, zmax=1, colorscale="RdBu", reversescale=True,
    colorbar=dict(title="r"), hovertemplate="%{y} × %{x}<br>r=%{z:.2f}<extra></extra>",
))
fig.update_layout(title="Корреляции признаков", width=1250, height=1250,
                  margin=dict(l=220, b=220))
fig.show()
logger.report_plotly(title="EDA", series="correlation_heatmap", figure=fig)

eda_pairs = (eda_corr.where(np.triu(np.ones(eda_corr.shape, dtype=bool), k=1))
             .stack().sort_values(key=abs, ascending=False))
print(f'''Самые сильные корреляции между признаками:
{eda_pairs.head(15).to_string()}''')
logger.report_table(title="EDA", series="top_correlations",
                    table_plot=eda_pairs.head(15).reset_index(name="r"))


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


Самые сильные корреляции между признаками:
has_user_reviews        user_rating                  0.999153
has_expert_credentials  has_sources_cited            0.935337
has_comparison          has_recommendation           0.911622
brand_mention_count     brand_in_title               0.899673
user_rating             has_author                  -0.884638
has_pros_cons           user_rating                 -0.842436
has_comparison          content_length_words         0.838381
brand_mentioned         has_product_specs            0.838082
has_pros_cons           has_author                   0.822668
user_rating             content_length_words        -0.822411
has_faq                 has_sources_cited            0.821339
brand_in_title          qtype_brand                  0.814864
url_has_year            content_freshness_2026       0.809508
has_recommendation      content_length_words         0.808359
content_freshness_2025  content_freshness_unknown   -0.800346


## Распределения и выбросы

In [6]:
eda_colors = {"Нет": "#4477AA", "Да": "#CC6677"}
eda_long = df.melt(id_vars=eda_target, value_vars=eda_multi,
                   var_name="Признак", value_name="Значение").dropna()
eda_long["Цитируется"] = eda_long[eda_target].map({0: "Нет", 1: "Да"})

fig = px.histogram(eda_long, x="Значение", color="Цитируется",
                   facet_col="Признак", facet_col_wrap=2,
                   color_discrete_map=eda_colors, barmode="overlay",
                   histnorm="probability", opacity=0.65,
                   title="Распределения признаков по классам")
for trace in fig.data:
    trace.bingroup = trace.xaxis

fig.update_xaxes(matches=None)
fig.update_layout(height=1500, width=1050)
fig.show()
logger.report_plotly(title="EDA", series="feature_distributions", figure=fig)

fig = px.box(eda_long, x="Цитируется", y="Значение", color="Цитируется",
             facet_col="Признак", facet_col_wrap=2,
             color_discrete_map=eda_colors, points="outliers",
             title="Boxplots признаков по классам")
fig.update_yaxes(matches=None)
fig.update_layout(height=1500, width=1050)
fig.show()
logger.report_plotly(title="EDA", series="feature_boxplots", figure=fig)


In [7]:
eda_prevalence = df.groupby(eda_target)[eda_binary].mean().T.sort_values(1)
fig = go.Figure()
for label, color in ((0, eda_colors["Нет"]), (1, eda_colors["Да"])):
    fig.add_bar(y=eda_prevalence.index, x=eda_prevalence[label], orientation="h",
                name=f"Цитируется: {label}", marker_color=color)
fig.update_layout(title="Доля страниц с бинарным признаком", barmode="group",
                  xaxis_title="Доля", height=1300, width=1050,
                  margin=dict(l=230))
fig.show()
logger.report_plotly(title="EDA", series="binary_prevalence", figure=fig)

fig = px.scatter(df.assign(citation=df[eda_target].map({0: "Нет", 1: "Да"})),
                 x="information_gain", y="product_focus_ratio",
                 color="citation", hover_data=["query", "domain"],
                 title="Объекты: уникальность информации и фокус на продукте",
                 color_discrete_map=eda_colors)
fig.update_traces(marker=dict(size=9, opacity=0.6))
fig.show()
logger.report_plotly(title="EDA", series="content_scatter", figure=fig)


## PCA: структура объектов и объяснённая дисперсия

Перед PCA пропуски заменяются медианой, а признаки стандартизируются, чтобы масштаб измерения не определял компоненты. Объяснённая дисперсия показывает сохранённую вариацию признаков, а не качество предсказания целевой переменной.


In [8]:
eda_scaled = StandardScaler().fit_transform(
    SimpleImputer(strategy="median").fit_transform(eda_numeric)
)
eda_pca = PCA().fit(eda_scaled)
eda_variance = eda_pca.explained_variance_ratio_
eda_cumulative = np.cumsum(eda_variance)
eda_components = np.arange(1, len(eda_variance) + 1)

eda_component_counts = {
    share: np.searchsorted(eda_cumulative, share) + 1
    for share in (0.80, 0.90, 0.95)
}

print(f'''80% дисперсии: {eda_component_counts[0.80]} из {len(eda_variance)} компонент
90% дисперсии: {eda_component_counts[0.90]} из {len(eda_variance)} компонент
95% дисперсии: {eda_component_counts[0.95]} из {len(eda_variance)} компонент''')

for share, count in eda_component_counts.items():
    logger.report_single_value(name=f"pca_components_{share:.0%}", value=int(count))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=eda_components, y=eda_variance,
                     name="Доля компоненты", marker_color="#4477AA"), secondary_y=False)
fig.add_trace(go.Scatter(x=eda_components, y=eda_cumulative,
                         name="Накопленная доля", line_color="#CC6677"), secondary_y=True)
fig.add_hline(y=0.90, line_dash="dash", line_color="gray", secondary_y=True)
fig.update_layout(title="PCA: объяснённая дисперсия", xaxis_title="Номер компоненты",
                  height=500, width=950)
fig.update_yaxes(title_text="Доля компоненты", secondary_y=False)
fig.update_yaxes(title_text="Накопленная доля", range=[0, 1.02], secondary_y=True)
fig.show()
logger.report_plotly(title="EDA", series="pca_explained_variance", figure=fig)

eda_scores = eda_pca.transform(eda_scaled)
eda_projection = df[["query", "domain", "query_type"]].copy()
eda_projection["citation"] = df[eda_target].map({0: "Нет", 1: "Да"})
eda_projection["PC1"] = eda_scores[:, 0]
eda_projection["PC2"] = eda_scores[:, 1]
fig = px.scatter(eda_projection, x="PC1", y="PC2", color="citation",
                 hover_data=["query", "domain", "query_type"],
                 color_discrete_map=eda_colors,
                 title="Страницы в пространстве первых двух компонент")
fig.update_traces(marker=dict(size=9, opacity=0.7))
fig.update_xaxes(title=f"PC1 ({eda_variance[0]:.1%} дисперсии)")
fig.update_yaxes(title=f"PC2 ({eda_variance[1]:.1%} дисперсии)")
fig.show()
logger.report_plotly(title="EDA", series="pca_projection", figure=fig)


80% дисперсии: 11 из 55 компонент
90% дисперсии: 16 из 55 компонент
95% дисперсии: 21 из 55 компонент


In [9]:
TARGET = "cited_in_generative_answer"
META = ["query", "query_type", "product_category", "source_url", "domain", "source_type"]

X = df.drop(columns=META + [TARGET]).astype(float)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

vt = VarianceThreshold(threshold=0.0).fit(X_train)
constant_cols = X_train.columns[~vt.get_support()].tolist()

X_train = X_train.loc[:, vt.get_support()].copy()
X_test = X_test[X_train.columns].copy()

corr = X_train.corr().abs()
feature_cols = []
to_drop = []

for col in X_train.columns:
    if (corr.loc[col, feature_cols] > 0.90).any():
        to_drop.append(col)
    else:
        feature_cols.append(col)

X_train = X_train[feature_cols]
X_test = X_test[feature_cols]

print(f'''Train: {len(y_train)} строк, Test: {len(y_test)} строк
Удалены по дисперсии: {constant_cols}
Удалены по корреляции: {to_drop}
Осталось признаков: {len(feature_cols)}''')

logger.report_single_value(name="train_rows", value=len(y_train))
logger.report_single_value(name="test_rows", value=len(y_test))
task.upload_artifact(
    name="pre_filter_features",
    artifact_object={
        "removed_constant": constant_cols,
        "removed_correlated": to_drop,
        "retained": feature_cols,
    },
)

Train: 156 строк, Test: 39 строк
Удалены по дисперсии: ['query_answered_directly', 'is_serp_link', 'source_type_serp_internal']
Удалены по корреляции: ['brand_in_title', 'has_recommendation', 'has_sources_cited']
Осталось признаков: 49

True

# Baseline

In [10]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

baseline_X = X.loc[y_train.index, feature_cols]

baseline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(
        C=1.0,
        l1_ratio=0.0,
        solver="saga",
        class_weight=None,
        fit_intercept=True,
        max_iter=5000,
        tol=1e-4,
        random_state=42,
    ),
)

scores = cross_validate(
    baseline,
    baseline_X,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision",
        "log_loss": "neg_log_loss",
        "f1": "f1",
    },
    error_score="raise",
)

auc = scores["test_roc_auc"]
pr_auc = scores["test_pr_auc"]
log_loss_cv = -scores["test_log_loss"]
f1 = scores["test_f1"]

print(f'''BASELINE — {len(feature_cols)} признаков, 3-fold CV
ROC-AUC:  {auc.mean():.3f} ± {auc.std():.3f}
PR-AUC:   {pr_auc.mean():.3f} ± {pr_auc.std():.3f}
Log Loss: {log_loss_cv.mean():.3f} ± {log_loss_cv.std():.3f}
F1:       {f1.mean():.3f} ± {f1.std():.3f}''')

task.connect(
    baseline.named_steps["logisticregression"].get_params(),
    name="baseline_logreg",
)

for name, values in {
    "roc_auc": auc,
    "pr_auc": pr_auc,
    "log_loss": log_loss_cv,
    "f1": f1,
}.items():
    logger.report_single_value(
        name=f"baseline_cv_{name}",
        value=float(values.mean()),
    )

BASELINE — 49 признаков, 3-fold CV
ROC-AUC:  0.716 ± 0.097
PR-AUC:   0.816 ± 0.060
Log Loss: 0.711 ± 0.183
F1:       0.682 ± 0.068


# RFECV

In [11]:
from itertools import combinations

from sklearn.base import clone
from sklearn.feature_selection import RFECV

X_rfecv = X.loc[y_train.index, feature_cols]

lr_params = dict(
    C=1.0,
    l1_ratio=0.0,
    solver="saga",
    class_weight=None,
    fit_intercept=True,
    max_iter=5000,
    tol=1e-4,
    random_state=42,
)

outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=43)

baseline_template = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(**lr_params),
)

selector = RFECV(
    estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(**lr_params),
    ),
    importance_getter="named_steps.logisticregression.coef_",
    step=5,
    min_features_to_select=5,
    cv=inner_cv,
    scoring="roc_auc",
)

rfecv_template = make_pipeline(
    SimpleImputer(strategy="median"),
    selector,
    StandardScaler(),
    LogisticRegression(**lr_params),
)

rows = []
selected_sets = []

for fold, (train_idx, val_idx) in enumerate(
    outer_cv.split(X_rfecv, y_train), start=1
):
    X_fit = X_rfecv.iloc[train_idx]
    X_val = X_rfecv.iloc[val_idx]
    y_fit = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    for name, template in (
        ("baseline", baseline_template),
        ("RFECV", rfecv_template),
    ):
        model = clone(template).fit(X_fit, y_fit)
        probabilities = model.predict_proba(X_val)[:, 1]

        n_features = len(feature_cols)
        if name == "RFECV":
            chosen = set(
                X_rfecv.columns[model.named_steps["rfecv"].support_]
            )
            selected_sets.append(chosen)
            n_features = len(chosen)

        auc = roc_auc_score(y_val, probabilities)
        loss = log_loss(y_val, probabilities)

        rows.append({
            "fold": fold,
            "model": name,
            "n_features": n_features,
            "roc_auc": auc,
            "log_loss": loss,
        })

        logger.report_scalar(
            title="RFECV experiment / ROC-AUC",
            series=name,
            value=float(auc),
            iteration=fold,
        )
        logger.report_scalar(
            title="RFECV experiment / Log Loss",
            series=name,
            value=float(loss),
            iteration=fold,
        )

fold_results = pd.DataFrame(rows)
summary = fold_results.groupby("model").agg(
    auc_mean=("roc_auc", "mean"),
    auc_std=("roc_auc", "std"),
    log_loss_mean=("log_loss", "mean"),
    mean_features=("n_features", "mean"),
).round(3)

selection_frequency = pd.DataFrame({
    "feature": feature_cols,
    "selected_folds": [
        sum(feature in chosen for chosen in selected_sets)
        for feature in feature_cols
    ],
}).sort_values(
    ["selected_folds", "feature"],
    ascending=[False, True],
)

mean_jaccard = np.mean([
    len(a & b) / len(a | b)
    for a, b in combinations(selected_sets, 2)
])

print(f'''Сравнение на внешних фолдах:
{summary.to_string()}

Число признаков RFECV по фолдам: {[len(s) for s in selected_sets]}
Среднее совпадение наборов (Jaccard): {mean_jaccard:.3f}

Частота отбора — топ-20:
{selection_frequency.head(20).to_string(index=False)}''')

logger.report_table(
    title="RFECV experiment",
    series="outer_fold_comparison",
    table_plot=fold_results,
)
logger.report_table(
    title="RFECV experiment",
    series="selection_frequency",
    table_plot=selection_frequency,
)
logger.report_single_value(
    name="rfecv_mean_jaccard",
    value=float(mean_jaccard),
)

rfecv_final = clone(rfecv_template).fit(X_rfecv, y_train)
final_selector = rfecv_final.named_steps["rfecv"]
selected_features = X_rfecv.columns[final_selector.support_].tolist()

curve = pd.DataFrame({
    "n_features": final_selector.cv_results_["n_features"],
    "mean_auc": final_selector.cv_results_["mean_test_score"],
    "std_auc": final_selector.cv_results_["std_test_score"],
})

fig = px.line(
    curve,
    x="n_features",
    y="mean_auc",
    error_y="std_auc",
    markers=True,
    title="RFECV: AUC внутренней CV по числу признаков",
)
fig.add_vline(x=final_selector.n_features_, line_dash="dash")
fig.show()

logger.report_plotly(
    title="RFECV experiment",
    series="inner_auc_by_feature_count",
    figure=fig,
)
task.upload_artifact(
    name="rfecv_selected_features",
    artifact_object={"features": selected_features},
)

print(f'''На всех обучающих данных RFECV выбрал
{final_selector.n_features_} из {len(feature_cols)} признаков:
{selected_features}''')

Сравнение на внешних фолдах:
          auc_mean  auc_std  log_loss_mean  mean_features
model                                                    
RFECV        0.682    0.101          0.741         17.333
baseline     0.716    0.119          0.711         49.000

Число признаков RFECV по фолдам: [9, 9, 34]
Среднее совпадение наборов (Jaccard): 0.258

Частота отбора — топ-20:
                  feature  selected_folds
     content_length_words               3
                  has_faq               3
             review_count               3
           url_has_rating               3
   content_freshness_2026               2
content_freshness_unknown               2
      product_focus_ratio               2
        readability_score               2
source_type_beauty_retail               2
         url_has_brand_pg               2
           url_has_how_to               2
      brand_mention_count               1
          brand_mentioned               1
   content_freshness_2025           

На всех обучающих данных RFECV выбрал
39 из 49 признаков:
['brand_mentioned', 'brand_mention_count', 'recommendation_sentiment', 'has_product_specs', 'has_numeric_data', 'has_comparison', 'user_rating', 'review_count', 'has_structured_headings', 'has_table_or_list', 'has_faq', 'product_focus_ratio', 'readability_score', 'has_author', 'has_expert_credentials', 'covers_intent_fully', 'information_gain', 'url_has_review', 'url_has_rating', 'url_has_comparison', 'url_has_article', 'url_has_catalog', 'url_has_brand_pg', 'url_has_year', 'url_has_how_to', 'source_type_marketplace', 'source_type_pharmacy', 'source_type_expert_dental', 'source_type_beauty_retail', 'source_type_review_platform', 'source_type_baby_expert', 'source_type_ecommerce_journal', 'content_freshness_2026', 'content_freshness_2025', 'content_freshness_unknown', 'qtype_recommendation', 'qtype_comparison', 'qtype_category', 'qtype_brand']


RFECV не улучшил прогноз на внешних фолдах: ROC-AUC снизился с 0,716 до 0,682, а Log Loss вырос с 0,711 до 0,741. Отбор также нестабилен: по фолдам осталось 9, 9 и 34 признака, среднее совпадение наборов по Жаккару — 0,258. Выбранные RFECV признаки нельзя считать статистически значимыми только на основании отбора.

# Optuna

In [12]:
X_opt = X.loc[y_train.index, feature_cols]

N_TRIALS = 50

outer_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=43,
)


def make_model(C=1.0, l1_ratio=0.0):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(
            C=C,
            l1_ratio=l1_ratio,
            solver="saga",
            class_weight=None,
            max_iter=5000,
            tol=1e-4,
            random_state=42,
        ),
    )


def optimize_model(X_fit, y_fit, seed):
    def objective(trial):
        C = trial.suggest_float("C", 1e-3, 10.0, log=True)
        l1_ratio = trial.suggest_categorical(
            "l1_ratio",
            [0.0, 0.5, 1.0],
        )

        scores = cross_val_score(
            make_model(C=C, l1_ratio=l1_ratio),
            X_fit,
            y_fit,
            cv=inner_cv,
            scoring="neg_log_loss",
            error_score="raise",
        )

        return -scores.mean()

    search = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )

    search.enqueue_trial({"C": 1.0, "l1_ratio": 0.0})
    search.optimize(objective, n_trials=N_TRIALS)

    return search


rows = []
outer_trials = []

for fold, (fit_idx, val_idx) in enumerate(
    outer_cv.split(X_opt, y_train),
    start=1,
):
    X_fit = X_opt.iloc[fit_idx]
    X_val = X_opt.iloc[val_idx]
    y_fit = y_train.iloc[fit_idx]
    y_val = y_train.iloc[val_idx]

    search = optimize_model(X_fit, y_fit, seed=42 + fold)

    trial_table = search.trials_dataframe()
    trial_table["outer_fold"] = fold
    outer_trials.append(trial_table)

    models = {
        "baseline": make_model(),
        "optuna": make_model(**search.best_params),
    }

    for name, candidate in models.items():
        fitted = clone(candidate).fit(X_fit, y_fit)
        probabilities = fitted.predict_proba(X_val)[:, 1]

        rows.append({
            "fold": fold,
            "model": name,
            "roc_auc": roc_auc_score(y_val, probabilities),
            "log_loss": log_loss(y_val, probabilities),
            "C": fitted.named_steps["logisticregression"].C,
            "l1_ratio": fitted.named_steps["logisticregression"].l1_ratio,
        })


fold_results = pd.DataFrame(rows)

summary = fold_results.groupby("model").agg(
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_std=("roc_auc", "std"),
    log_loss_mean=("log_loss", "mean"),
    log_loss_std=("log_loss", "std"),
).round(3)

comparison = fold_results.pivot(
    index="fold",
    columns="model",
    values=["roc_auc", "log_loss"],
)

auc_delta = (
    comparison["roc_auc"]["optuna"]
    - comparison["roc_auc"]["baseline"]
)

loss_delta = (
    comparison["log_loss"]["optuna"]
    - comparison["log_loss"]["baseline"]
)

print(f'''Внешняя кросс-валидация:
{summary.to_string()}

Изменение Optuna относительно бейзлайна по фолдам:
ROC-AUC:  {auc_delta.round(3).tolist()}
Log Loss: {loss_delta.round(3).tolist()}

Среднее изменение:
ROC-AUC:  {auc_delta.mean():+.3f}
Log Loss: {loss_delta.mean():+.3f}''')

logger.report_table(
    title="Optuna",
    series="outer_fold_comparison",
    table_plot=fold_results,
)

logger.report_table(
    title="Optuna",
    series="outer_trial_history",
    table_plot=pd.concat(outer_trials, ignore_index=True),
)

for model_name, metrics in summary.iterrows():
    for metric in ("roc_auc_mean", "log_loss_mean"):
        logger.report_single_value(
            name=f"{model_name}_outer_{metric}",
            value=float(metrics[metric]),
        )


study = optimize_model(X_opt, y_train, seed=42)

optimized_model = make_model(**study.best_params)
optimized_model.fit(X_opt, y_train)

logger.report_table(
    title="Optuna",
    series="final_trial_history",
    table_plot=study.trials_dataframe(),
)

logger.report_single_value(
    name="optuna_best_inner_cv_log_loss",
    value=float(study.best_value),
)

task.upload_artifact(
    name="optuna_best_params",
    artifact_object=study.best_params,
)

print(f'''Финальный подбор на обучающей части:
CV Log Loss: {study.best_value:.3f}
Параметры:  {study.best_params}''')

Внешняя кросс-валидация:
          roc_auc_mean  roc_auc_std  log_loss_mean  log_loss_std
model                                                           
baseline         0.716        0.119          0.711         0.224
optuna           0.688        0.084          0.624         0.027

Изменение Optuna относительно бейзлайна по фолдам:
ROC-AUC:  [-0.057, -0.045, 0.02]
Log Loss: [0.085, -0.041, -0.306]

Среднее изменение:
ROC-AUC:  -0.027
Log Loss: -0.087


Финальный подбор на обучающей части:
CV Log Loss: 0.622
Параметры:  {'C': 0.0353294307287589, 'l1_ratio': 0.0}


In [13]:
from sklearn.model_selection import learning_curve

curve_rows = []
train_sizes = np.linspace(0.35, 1.0, 6)

for model_name, candidate in {
    "baseline": make_model(),
    "optuna": make_model(**study.best_params),
}.items():
    for metric, scoring in {
        "ROC-AUC": "roc_auc",
        "Log Loss": "neg_log_loss",
    }.items():
        sizes, train_scores, valid_scores = learning_curve(
            candidate,
            X_opt,
            y_train,
            train_sizes=train_sizes,
            cv=outer_cv,
            scoring=scoring,
            shuffle=True,
            random_state=42,
            error_score="raise",
        )

        if metric == "Log Loss":
            train_scores = -train_scores
            valid_scores = -valid_scores

        for size, train_fold_scores, valid_fold_scores in zip(
            sizes,
            train_scores,
            valid_scores,
        ):
            for split, scores in {
                "Обучение": train_fold_scores,
                "Проверка": valid_fold_scores,
            }.items():
                curve_rows.append({
                    "model": model_name,
                    "metric": metric,
                    "n_train": int(size),
                    "split": split,
                    "mean": float(scores.mean()),
                    "std": float(scores.std(ddof=1)),
                })

learning_curves = pd.DataFrame(curve_rows)

for metric in ("ROC-AUC", "Log Loss"):
    fig = make_subplots(
        rows=1,
        cols=2,
        shared_yaxes=True,
        subplot_titles=("Бейзлайн", "Optuna"),
    )

    for col, model_name in enumerate(("baseline", "optuna"), start=1):
        for split, color in {
            "Обучение": "#4477AA",
            "Проверка": "#CC6677",
        }.items():
            part = learning_curves[
                (learning_curves["metric"] == metric)
                & (learning_curves["model"] == model_name)
                & (learning_curves["split"] == split)
            ]

            fig.add_trace(
                go.Scatter(
                    x=part["n_train"],
                    y=part["mean"],
                    mode="lines+markers",
                    name=split,
                    legendgroup=split,
                    showlegend=(col == 1),
                    line=dict(color=color),
                    error_y=dict(
                        type="data",
                        array=part["std"],
                        visible=True,
                    ),
                ),
                row=1,
                col=col,
            )

    fig.update_layout(
        title=f"Кривая обучения: {metric}",
        width=1050,
        height=450,
    )
    fig.update_xaxes(title_text="Число обучающих страниц")
    fig.update_yaxes(title_text=metric, col=1)

    fig.show()

    logger.report_plotly(
        title="Learning curves",
        series=metric,
        figure=fig,
    )

logger.report_table(
    title="Learning curves",
    series="scores_by_train_size",
    table_plot=learning_curves,
)


trials = study.trials_dataframe(
    attrs=("number", "value", "state"),
)

trials = trials[trials["state"] == "COMPLETE"].copy()
trials["trial"] = trials["number"] + 1
trials["best_so_far"] = trials["value"].cummin()

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=trials["trial"],
        y=trials["value"],
        mode="markers",
        name="Log Loss попытки",
    )
)

fig.add_trace(
    go.Scatter(
        x=trials["trial"],
        y=trials["best_so_far"],
        mode="lines+markers",
        name="Лучший Log Loss к этому шагу",
    )
)

fig.update_layout(
    title="Ход оптимизации Optuna",
    xaxis_title="Номер попытки",
    yaxis_title="CV Log Loss",
    width=900,
    height=450,
)

fig.show()

logger.report_plotly(
    title="Optuna",
    series="optimization_history",
    figure=fig,
)

logger.report_table(
    title="Optuna",
    series="optimization_history_data",
    table_plot=trials,
)

# Bootstrap

In [14]:
N_BOOTSTRAPS = 500
rng = np.random.default_rng(42)

queries = df.loc[y_train.index, "query"].to_numpy()
unique_queries = pd.unique(queries)

query_rows = [
    np.flatnonzero(queries == query)
    for query in unique_queries
]

reference_scaler = optimized_model.named_steps["standardscaler"]
reference_lr = optimized_model.named_steps["logisticregression"]

reference_scale = reference_scaler.scale_
reference_coef = reference_lr.coef_[0]

bootstrap_coef = []

for _ in range(N_BOOTSTRAPS):
    while True:
        sampled_queries = rng.integers(
            0,
            len(query_rows),
            size=len(query_rows),
        )

        sampled_rows = np.concatenate([
            query_rows[i] for i in sampled_queries
        ])

        if y_train.iloc[sampled_rows].nunique() == 2:
            break

    fitted = make_model(**study.best_params).fit(
        X_opt.iloc[sampled_rows],
        y_train.iloc[sampled_rows],
    )

    fitted_scaler = fitted.named_steps["standardscaler"]
    fitted_lr = fitted.named_steps["logisticregression"]

    # Переводим коэффициенты к масштабу полного train:
    # в каждом бутстрапе StandardScaler обучается заново.
    coef_on_reference_scale = (
        fitted_lr.coef_[0]
        / fitted_scaler.scale_
        * reference_scale
    )

    bootstrap_coef.append(coef_on_reference_scale)

bootstrap_coef = np.asarray(bootstrap_coef)

coef_low, coef_high = np.quantile(
    bootstrap_coef,
    [0.025, 0.975],
    axis=0,
)

sign_stability = (
    np.sign(bootstrap_coef) == np.sign(reference_coef)
).mean(axis=0)

top_10_indices = np.argsort(
    -np.abs(bootstrap_coef),
    axis=1,
)[:, :10]

top_10_frequency = (
    np.bincount(
        top_10_indices.ravel(),
        minlength=len(feature_cols),
    )
    / N_BOOTSTRAPS
)

is_binary = np.array([
    set(X_opt[col].dropna().unique()).issubset({0, 1})
    for col in feature_cols
])

# Для бинарного признака: переход 0 → 1.
# Для непрерывного: увеличение на 1 SD полного train.
effect_multiplier = np.where(
    is_binary,
    1 / reference_scale,
    1.0,
)

effect_coef = reference_coef * effect_multiplier
bootstrap_effect = bootstrap_coef * effect_multiplier

effect_low, effect_high = np.quantile(
    bootstrap_effect,
    [0.025, 0.975],
    axis=0,
)

interpretation = pd.DataFrame({
    "feature": feature_cols,
    "coef_per_sd": reference_coef,
    "coef_low": coef_low,
    "coef_high": coef_high,
    "sign_stability": sign_stability,
    "top_10_frequency": top_10_frequency,
    "effect": np.where(is_binary, "0 → 1", "+1 SD"),
    "odds_ratio": np.exp(effect_coef),
    "or_low": np.exp(effect_low),
    "or_high": np.exp(effect_high),
})

interpretation["stable"] = (
    (interpretation["sign_stability"] >= 0.95)
    & (
        (interpretation["coef_low"] > 0)
        | (interpretation["coef_high"] < 0)
    )
)

interpretation = interpretation.sort_values(
    "coef_per_sd",
    key=lambda values: values.abs(),
    ascending=False,
)

top = interpretation.head(20).sort_values("coef_per_sd")

fig = go.Figure()

for row in top.itertuples():
    fig.add_trace(
        go.Scatter(
            x=[row.coef_low, row.coef_high],
            y=[row.feature, row.feature],
            mode="lines",
            line=dict(
                color="#4477AA" if row.stable else "#BBBBBB",
                width=3,
            ),
            showlegend=False,
            hoverinfo="skip",
        )
    )

fig.add_trace(
    go.Scatter(
        x=top["coef_per_sd"],
        y=top["feature"],
        mode="markers",
        marker=dict(
            size=9,
            color=[
                "#4477AA" if stable else "#BBBBBB"
                for stable in top["stable"]
            ],
        ),
        name="Коэффициент модели",
    )
)

fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.update_layout(
    title="Устойчивость коэффициентов: бутстрап по запросам",
    xaxis_title="Коэффициент при изменении на 1 SD",
    height=750,
    width=1050,
    margin=dict(l=260),
)

fig.show()

logger.report_plotly(
    title="Model interpretation",
    series="coefficient_stability",
    figure=fig,
)

logger.report_table(
    title="Model interpretation",
    series="bootstrap_coefficients",
    table_plot=interpretation,
)

task.upload_artifact(
    name="bootstrap_coefficients",
    artifact_object=interpretation,
)

print(f'''Бутстрап по {len(unique_queries)} запросам,
повторов: {N_BOOTSTRAPS}

Устойчивый знак и интервал без нуля:
{interpretation["stable"].sum()} из {len(interpretation)} признаков

Топ-15 по модулю коэффициента:
{interpretation.head(15).to_string(index=False)}''')

Бутстрап по 26 запросам,
повторов: 500

Устойчивый знак и интервал без нуля:
17 из 49 признаков

Топ-15 по модулю коэффициента:
                      feature  coef_per_sd  coef_low  coef_high  sign_stability  top_10_frequency effect  odds_ratio   or_low  or_high  stable
               url_has_rating    -0.209030 -0.318726  -0.086116           1.000             0.880  0 → 1    0.637410 0.503246 0.830662    True
                 review_count    -0.197193 -0.291673  -0.011071           0.984             0.784  +1 SD    0.821032 0.747013 0.988990    True
               url_has_how_to     0.183424  0.117560   0.253312           1.000             0.884  0 → 1    1.752156 1.432552 2.169589    True
               qtype_category     0.180232  0.117945   0.279732           0.996             0.922  0 → 1    1.579819 1.348870 2.033535    True
         qtype_recommendation    -0.159915 -0.268414  -0.055032           1.000             0.620  0 → 1    0.711127 0.564281 0.889303    True
    source_typ

### Устойчивость и интерпретация коэффициентов

Бутстрап по 26 запросам (500 повторов) показал устойчивое направление связи у 17 из 49 признаков. Среди них положительные коэффициенты у `url_has_how_to`, `has_faq` и `information_gain`, отрицательные — у `url_has_rating` и `review_count`. Удалённый после очистки `query_answered_directly` больше не интерпретируется. Интервалы описывают устойчивость при уже выбранных признаках и регуляризации, а не статистическую значимость или причинное влияние.

In [15]:
from sklearn.model_selection import StratifiedGroupKFold

query_groups = df.loc[y_train.index, "query"]

group_outer_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

group_inner_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=43,
)

group_rows = []

for fold, (fit_idx, val_idx) in enumerate(
    group_outer_cv.split(X_opt, y_train, query_groups),
    start=1,
):
    X_fit = X_opt.iloc[fit_idx]
    X_val = X_opt.iloc[val_idx]
    y_fit = y_train.iloc[fit_idx]
    y_val = y_train.iloc[val_idx]
    groups_fit = query_groups.iloc[fit_idx]

    def group_objective(trial):
        C = trial.suggest_float("C", 1e-3, 10.0, log=True)
        l1_ratio = trial.suggest_categorical(
            "l1_ratio",
            [0.0, 0.5, 1.0],
        )

        losses = []

        for inner_fit, inner_val in group_inner_cv.split(
            X_fit,
            y_fit,
            groups_fit,
        ):
            candidate = make_model(C=C, l1_ratio=l1_ratio)
            candidate.fit(
                X_fit.iloc[inner_fit],
                y_fit.iloc[inner_fit],
            )

            probabilities = candidate.predict_proba(
                X_fit.iloc[inner_val]
            )[:, 1]

            losses.append(
                log_loss(y_fit.iloc[inner_val], probabilities)
            )

        return float(np.mean(losses))

    group_study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42 + fold),
    )

    group_study.enqueue_trial({"C": 1.0, "l1_ratio": 0.0})
    group_study.optimize(group_objective, n_trials=30)

    for name, params in {
        "baseline": {},
        "optuna": group_study.best_params,
    }.items():
        candidate = make_model(**params)
        candidate.fit(X_fit, y_fit)

        probabilities = candidate.predict_proba(X_val)[:, 1]

        group_rows.append({
            "fold": fold,
            "model": name,
            "train_queries": groups_fit.nunique(),
            "val_queries": query_groups.iloc[val_idx].nunique(),
            "roc_auc": roc_auc_score(y_val, probabilities),
            "log_loss": log_loss(y_val, probabilities),
            "C": candidate.named_steps["logisticregression"].C,
            "l1_ratio": (
                candidate.named_steps["logisticregression"].l1_ratio
            ),
        })

group_results = pd.DataFrame(group_rows)

group_summary = group_results.groupby("model").agg(
    roc_auc=("roc_auc", "mean"),
    log_loss=("log_loss", "mean"),
).round(3)

print(f'''Проверка на новых запросах:
{group_summary.to_string()}

Результаты по фолдам:
{group_results.to_string(index=False)}''')

logger.report_table(
    title="Group CV",
    series="query_fold_comparison",
    table_plot=group_results,
)

task.upload_artifact(
    name="group_cv_results",
    artifact_object=group_results,
)

Проверка на новых запросах:
          roc_auc  log_loss
model                      
baseline    0.755     0.673
optuna      0.760     0.588

Результаты по фолдам:
 fold    model  train_queries  val_queries  roc_auc  log_loss        C  l1_ratio
    1 baseline             18            8 0.720788  0.852781 1.000000       0.0
    1   optuna             18            8 0.734375  0.603755 0.052776       0.0
    2 baseline             17            9 0.781250  0.539705 1.000000       0.0
    2   optuna             17            9 0.793155  0.581685 0.027368       0.0
    3 baseline             17            9 0.762755  0.625301 1.000000       0.0
    3   optuna             17            9 0.752551  0.577871 0.088695       0.0


True

На групповой CV с новыми запросами Optuna дала средний Log Loss 0,588 против 0,673 у бейзлайна и ROC-AUC 0,760 против 0,755. Log Loss улучшился на двух из трёх фолдов, ROC-AUC также вырос на двух; различия между фолдами заметны. Поэтому эти три разбиения поддерживают выбор регуляризованной модели для дальнейшего анализа, но не доказывают устойчивое улучшение на любых новых запросах.

In [16]:
fold_coefficients = []

for fold, (fit_idx, _) in enumerate(
    group_outer_cv.split(X_opt, y_train, query_groups),
    start=1,
):
    selected = group_results.loc[
        (group_results["fold"] == fold)
        & (group_results["model"] == "optuna")
    ].iloc[0]

    fitted = make_model(
        C=float(selected["C"]),
        l1_ratio=float(selected["l1_ratio"]),
    ).fit(
        X_opt.iloc[fit_idx],
        y_train.iloc[fit_idx],
    )

    fold_scaler = fitted.named_steps["standardscaler"]
    fold_lr = fitted.named_steps["logisticregression"]

    # Общий масштаб: изменение признака на 1 SD полного train.
    coefficients = (
        fold_lr.coef_[0]
        / fold_scaler.scale_
        * reference_scale
    )

    fold_coefficients.append(coefficients)

fold_coefficients = pd.DataFrame(
    fold_coefficients,
    index=["fold_1", "fold_2", "fold_3"],
    columns=feature_cols,
)

group_sign_folds = (
    np.sign(fold_coefficients.to_numpy())
    == np.sign(reference_coef)
).sum(axis=0)

group_top_10 = np.argsort(
    -np.abs(fold_coefficients.to_numpy()),
    axis=1,
)[:, :10]

group_top_10_frequency = (
    np.bincount(
        group_top_10.ravel(),
        minlength=len(feature_cols),
    )
    / len(fold_coefficients)
)

coefficient_checks = interpretation.set_index("feature").loc[
    feature_cols,
    [
        "coef_per_sd",
        "coef_low",
        "coef_high",
        "sign_stability",
        "stable",
    ],
].copy()

for fold in fold_coefficients.index:
    coefficient_checks[fold] = fold_coefficients.loc[fold]

coefficient_checks["group_sign_folds"] = group_sign_folds
coefficient_checks["group_top_10_frequency"] = (
    group_top_10_frequency
)

coefficient_checks["stable_in_both"] = (
    coefficient_checks["stable"]
    & (coefficient_checks["group_sign_folds"] == 3)
)

coefficient_checks = coefficient_checks.sort_values(
    "coef_per_sd",
    key=lambda values: values.abs(),
    ascending=False,
)

rank_correlation = (
    fold_coefficients.abs().T.corr(method="spearman")
)

plot_features = coefficient_checks.head(20).index

plot_coefficients = pd.concat([
    pd.Series(
        reference_coef,
        index=feature_cols,
        name="full_train",
    ).to_frame().T,
    fold_coefficients,
])

fig = go.Figure(
    go.Heatmap(
        z=plot_coefficients[plot_features].T.to_numpy(),
        x=plot_coefficients.index,
        y=plot_features,
        colorscale="RdBu",
        reversescale=True,
        zmid=0,
        colorbar_title="Коэффициент",
    )
)

fig.update_layout(
    title="Коэффициенты на разных наборах запросов",
    width=900,
    height=750,
    margin=dict(l=260),
)

fig.show()

logger.report_plotly(
    title="Model interpretation",
    series="group_fold_coefficients",
    figure=fig,
)

logger.report_table(
    title="Model interpretation",
    series="coefficient_stability_combined",
    table_plot=coefficient_checks.reset_index(),
)

logger.report_table(
    title="Model interpretation",
    series="coefficient_rank_correlation",
    table_plot=rank_correlation,
)

task.upload_artifact(
    name="coefficient_stability_combined",
    artifact_object=coefficient_checks.reset_index(),
)

print(f'''Совпадение знака с полной моделью на всех 3 фолдах:
{(coefficient_checks["group_sign_folds"] == 3).sum()} из {len(feature_cols)}

Устойчивы и в бутстрапе, и на групповых фолдах:
{coefficient_checks["stable_in_both"].sum()} из {len(feature_cols)}

Признаки, прошедшие обе проверки:
{coefficient_checks.loc[
    coefficient_checks["stable_in_both"],
    ["coef_per_sd", "sign_stability", "group_sign_folds"]
].to_string()}

Сходство рангов по модулю коэффициента:
{rank_correlation.round(3).to_string()}''')

Совпадение знака с полной моделью на всех 3 фолдах:
36 из 49

Устойчивы и в бутстрапе, и на групповых фолдах:
17 из 49

Признаки, прошедшие обе проверки:
                           coef_per_sd  sign_stability  group_sign_folds
feature                                                                 
url_has_rating               -0.209030           1.000                 3
review_count                 -0.197193           0.984                 3
url_has_how_to                0.183424           1.000                 3
qtype_category                0.180232           0.996                 3
qtype_recommendation         -0.159915           1.000                 3
has_faq                       0.154592           1.000                 3
url_has_brand_pg              0.144916           0.992                 3
source_type_marketplace      -0.132945           0.976                 3
source_type_baby_expert       0.129918           1.000                 3
content_freshness_2026        0.108988     

# Финальная модель

In [17]:
model = make_model(**study.best_params)
X_test_raw = X.loc[y_test.index, feature_cols]

y_proba_cv = cross_val_predict(
    model,
    X_opt,
    y_train,
    cv=outer_cv,
    method="predict_proba",
)[:, 1]
y_pred_cv = (y_proba_cv >= 0.5).astype(int)

print(f'''TRAIN — CV с фиксированными параметрами:
ROC-AUC:  {roc_auc_score(y_train, y_proba_cv):.4f}
Log Loss: {log_loss(y_train, y_proba_cv):.4f}

{classification_report(y_train, y_pred_cv, target_names=["Не цитируется", "Цитируется"])}''')

model.fit(X_opt, y_train)
y_proba_test = model.predict_proba(X_test_raw)[:, 1]
y_pred_test = (y_proba_test >= 0.5).astype(int)

print(f'''TEST:
ROC-AUC:  {roc_auc_score(y_test, y_proba_test):.4f}
PR-AUC:   {average_precision_score(y_test, y_proba_test):.4f}
Log Loss: {log_loss(y_test, y_proba_test):.4f}

{classification_report(y_test, y_pred_test, target_names=["Не цитируется", "Цитируется"])}''')

logger.report_single_value(
    name="logreg_cv_auc",
    value=float(roc_auc_score(y_train, y_proba_cv)),
)
logger.report_single_value(
    name="logreg_test_auc",
    value=float(roc_auc_score(y_test, y_proba_test)),
)
logger.report_single_value(
    name="logreg_test_log_loss",
    value=float(log_loss(y_test, y_proba_test)),
)
logger.report_table(
    title="Logistic Regression",
    series="test_classification_report",
    table_plot=pd.DataFrame(
        classification_report(y_test, y_pred_test, output_dict=True)
    ).T,
)

coef = model.named_steps["logisticregression"].coef_[0]
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coef": coef,
    "OR": np.exp(coef),
}).assign(
    abs_coef=lambda data: data["coef"].abs()
).sort_values("abs_coef", ascending=False)

logger.report_table(
    title="Logistic Regression",
    series="coefficients_per_sd",
    table_plot=coef_df,
)
task.upload_artifact(
    name="logreg_coefficients_per_sd",
    artifact_object=coef_df,
)

print(f'''ТОП-15 коэффициентов (на 1 SD):
{coef_df.head(15)[["feature", "coef", "OR"]].to_string(index=False)}''')


TRAIN — CV с фиксированными параметрами:
ROC-AUC:  0.6913
Log Loss: 0.6098

               precision    recall  f1-score   support

Не цитируется       0.54      0.57      0.56        68
   Цитируется       0.65      0.62      0.64        88

     accuracy                           0.60       156
    macro avg       0.60      0.60      0.60       156
 weighted avg       0.61      0.60      0.60       156

TEST:
ROC-AUC:  0.6043
PR-AUC:   0.7031
Log Loss: 0.6914

               precision    recall  f1-score   support

Не цитируется       0.59      0.59      0.59        17
   Цитируется       0.68      0.68      0.68        22

     accuracy                           0.64        39
    macro avg       0.64      0.64      0.64        39
 weighted avg       0.64      0.64      0.64        39



ТОП-15 коэффициентов (на 1 SD):
                      feature      coef       OR
               url_has_rating -0.209030 0.811371
                 review_count -0.197193 0.821032
               url_has_how_to  0.183424 1.201324
               qtype_category  0.180232 1.197495
         qtype_recommendation -0.159915 0.852217
    source_type_beauty_retail  0.155298 1.168006
                      has_faq  0.154592 1.167182
             url_has_brand_pg  0.144916 1.155942
      source_type_marketplace -0.132945 0.875513
      source_type_baby_expert  0.129918 1.138735
source_type_ecommerce_journal  0.127047 1.135470
       content_freshness_2026  0.108988 1.115149
  source_type_review_platform -0.100099 0.904748
    source_type_expert_dental  0.095165 1.099840
             information_gain  0.092545 1.096962


# Визуализация логистической регрессии и анализ ошибок

In [18]:
from pathlib import Path

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

plot_dir = Path("plots")
plot_dir.mkdir(exist_ok=True)


def save_figure(figure, filename, series):
    logger.report_plotly(
        title="Logistic Regression plots",
        series=series,
        figure=figure,
    )
    figure.write_html(plot_dir / filename, include_plotlyjs="cdn")
    figure.show()


# Коэффициенты: цвет показывает согласие бутстрапа и групповой CV.
top = coefficient_checks.head(20).sort_values("coef_per_sd")
fig = go.Figure(go.Bar(
    x=top["coef_per_sd"],
    y=top.index,
    orientation="h",
    marker_color=[
        "#4477AA" if stable else "#BBBBBB"
        for stable in top["stable_in_both"]
    ],
))
fig.add_vline(x=0, line_color="black")
fig.update_layout(
    title="Логистическая регрессия: коэффициенты на 1 SD",
    xaxis_title="Стандартизованный коэффициент",
    height=700,
    margin=dict(l=280),
)
save_figure(fig, "01_logreg_coefficients.html", "coefficients")


# ROC и PR: train CV здесь диагностическая оценка с фиксированными параметрами.
fig = go.Figure()
for label, target, probabilities in (
    ("Train CV", y_train, y_proba_cv),
    ("Test", y_test, y_proba_test),
):
    fpr, tpr, _ = roc_curve(target, probabilities)
    fig.add_trace(go.Scatter(
        x=fpr,
        y=tpr,
        name=f"{label}: AUC={roc_auc_score(target, probabilities):.3f}",
    ))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], name="Случайный прогноз",
    line=dict(dash="dash", color="gray"),
))
fig.update_layout(
    title="ROC: логистическая регрессия",
    xaxis_title="FPR",
    yaxis_title="TPR",
    height=480,
    width=700,
)
save_figure(fig, "02_roc_curve.html", "roc_curve")

fig = go.Figure()
for label, target, probabilities in (
    ("Train CV", y_train, y_proba_cv),
    ("Test", y_test, y_proba_test),
):
    precision, recall, _ = precision_recall_curve(target, probabilities)
    fig.add_trace(go.Scatter(
        x=recall,
        y=precision,
        name=f"{label}: AP={average_precision_score(target, probabilities):.3f}",
    ))
fig.add_hline(
    y=float(y_test.mean()),
    line_dash="dash",
    line_color="gray",
    annotation_text="Доля положительного класса в тесте",
)
fig.update_layout(
    title="Precision–Recall: логистическая регрессия",
    xaxis_title="Полнота",
    yaxis_title="Точность",
    height=480,
    width=700,
)
save_figure(fig, "03_precision_recall.html", "precision_recall")


# Ошибки на тесте при заранее фиксированном пороге 0.5.
cm = confusion_matrix(y_test, y_pred_test)
fig = go.Figure(go.Heatmap(
    z=cm,
    x=["Прогноз 0", "Прогноз 1"],
    y=["Факт 0", "Факт 1"],
    text=cm,
    texttemplate="%{text}",
    colorscale="Blues",
    showscale=False,
))
fig.update_layout(
    title="Матрица ошибок: тест, порог 0.5",
    height=420,
    width=500,
)
save_figure(fig, "04_confusion_matrix.html", "confusion_matrix")


# Калибровка: всего 39 тестовых страниц, поэтому кривая теста шумная.
fig = go.Figure()
for label, target, probabilities in (
    ("Train CV", y_train, y_proba_cv),
    ("Test", y_test, y_proba_test),
):
    observed, predicted = calibration_curve(
        target,
        probabilities,
        n_bins=5,
        strategy="quantile",
    )
    brier = brier_score_loss(target, probabilities)
    fig.add_trace(go.Scatter(
        x=predicted,
        y=observed,
        mode="lines+markers",
        name=f"{label}: Brier={brier:.3f}",
    ))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], name="Идеальная калибровка",
    line=dict(dash="dash", color="gray"),
))
fig.update_layout(
    title="Калибровка вероятностей",
    xaxis_title="Средняя предсказанная вероятность",
    yaxis_title="Наблюдаемая доля цитирований",
    height=480,
    width=700,
)
save_figure(fig, "05_calibration.html", "calibration")
logger.report_single_value(
    name="logreg_test_brier",
    value=float(brier_score_loss(y_test, y_proba_test)),
)


# Диагностика по сегментам оценивает одну и ту же модель, без переобучения внутри сегмента.
prediction_parts = []
for split, target, probabilities in (
    ("train_cv", y_train, y_proba_cv),
    ("test", y_test, y_proba_test),
):
    part = df.loc[
        target.index,
        ["query", "source_url", "product_category", "query_type"],
    ].copy()
    part["actual"] = target.to_numpy()
    part["probability"] = probabilities
    part["predicted"] = (probabilities >= 0.5).astype(int)
    part["correct"] = part["actual"] == part["predicted"]
    safe_probabilities = np.clip(probabilities, 1e-12, 1 - 1e-12)
    part["log_loss"] = -(
        part["actual"] * np.log(safe_probabilities)
        + (1 - part["actual"]) * np.log1p(-safe_probabilities)
    )
    part["split"] = split
    prediction_parts.append(part)

predictions = pd.concat(prediction_parts, ignore_index=True)
segment_rows = []
for column, dimension in (
    ("product_category", "Категория товара"),
    ("query_type", "Тип запроса"),
):
    for (split, segment), part in predictions.groupby(["split", column]):
        segment_rows.append({
            "split": split,
            "dimension": dimension,
            "segment": segment,
            "n": len(part),
            "positive_share": part["actual"].mean(),
            "roc_auc": (
                roc_auc_score(part["actual"], part["probability"])
                if part["actual"].nunique() == 2 else np.nan
            ),
            "log_loss": part["log_loss"].mean(),
            "error_rate_at_0.5": 1 - part["correct"].mean(),
        })

segment_scores = pd.DataFrame(segment_rows)
fig = px.bar(
    segment_scores,
    x="segment",
    y="log_loss",
    color="split",
    facet_col="dimension",
    barmode="group",
    text="n",
    title="Где модель ошибается: Log Loss по сегментам (подписи — n)",
)
fig.update_layout(height=500, width=1050)
fig.update_xaxes(tickangle=-25)
save_figure(fig, "06_segment_log_loss.html", "segment_log_loss")

worst_test_errors = predictions.loc[
    (predictions["split"] == "test") & (~predictions["correct"])
].nlargest(10, "log_loss")

logger.report_table(
    title="Logistic Regression diagnostics",
    series="segment_scores",
    table_plot=segment_scores,
)
logger.report_table(
    title="Logistic Regression diagnostics",
    series="worst_test_errors",
    table_plot=worst_test_errors,
)
task.upload_artifact(
    name="logreg_test_predictions",
    artifact_object=predictions.loc[predictions["split"] == "test"],
)

print(f'''Log Loss по сегментам:
{segment_scores.sort_values(["split", "dimension", "log_loss"], ascending=[True, True, False]).to_string(index=False)}

Самые уверенные ошибки на тесте:
{worst_test_errors[["query", "product_category", "query_type", "actual", "probability", "log_loss"]].to_string(index=False)}''')


# Диагностический подбор порога только по train CV, без использования теста.
thresholds = np.arange(0.1, 0.91, 0.01)
f1s = [
    f1_score(y_train, (y_proba_cv >= threshold).astype(int), zero_division=0)
    for threshold in thresholds
]
best_t = thresholds[np.argmax(f1s)]

fig = go.Figure(go.Scatter(
    x=thresholds,
    y=f1s,
    mode="lines",
    name="F1 на train CV",
))
fig.add_vline(x=best_t, line_dash="dash")
fig.update_layout(
    title=f"F1 по порогу на train CV: максимум при {best_t:.2f}",
    xaxis_title="Порог",
    yaxis_title="F1",
    height=400,
    width=700,
)
save_figure(fig, "07_threshold_train_cv.html", "threshold_train_cv")

print(f'''Графики сохранены в {plot_dir.resolve()}''')


Log Loss по сегментам:
   split        dimension          segment  n  positive_share  roc_auc  log_loss  error_rate_at_0.5
    test Категория товара        Oral Care 18        0.611111 0.467532  0.795717           0.500000
    test Категория товара        Baby Care 13        0.538462 0.595238  0.670238           0.384615
    test Категория товара        Hair Care  8        0.500000 1.000000  0.491149           0.000000
    test      Тип запроса Рекомендационный 15        0.600000 0.444444  0.812950           0.466667
    test      Тип запроса    Сравнительный 11        0.272727 0.625000  0.757186           0.454545
    test      Тип запроса     Категорийный  7        0.857143 0.833333  0.566166           0.285714
    test      Тип запроса        Брендовый  6        0.666667 1.000000  0.413121           0.000000
train_cv Категория товара        Hair Care 26        0.576923 0.684848  0.632603           0.423077
train_cv Категория товара        Oral Care 69        0.594203 0.655488  0.629

Графики сохранены в /Users/savin/Documents/Programming/Cup IT 2026/notebooks/plots


In [19]:
task.close()
print("Результаты записаны в сессию ClearML.")

Результаты записаны в сессию ClearML.
